# Adaptive Uniform Tanh Scaling Experiment

**Hypothesis:** By tracking strict min/max of pre-activations (G_min, G_max) and scaling them to a target range, we can:
1. Ensure tanh outputs are uniformly distributed by default (good exploration)
2. Allow outputs to reach near ±1 when learning genuinely pushes to extremes
3. Prevent accidental saturation from numerical instability

## Key Mechanism

```
Track: G_min, G_max (observed pre-activation range per dimension)
Scale: Map [G_min, G_max] → [-4, +4]
Result: tanh(±4) ≈ ±0.999, uniform output by default
```

## Architecture

| Network | Hidden Activations | Pre-Output | Output Activation | Architecture |
|---------|-------------------|------------|-------------------|---------------|
| Actor   | ReLU | **Adaptive Scaling** | tanh | 7->64->32->AdaptiveScale->tanh->3 |
| Critic  | ReLU | None | linear | 510->512->128->1 |

## Learning Rate Order (High to Low)
Experiments run from **HIGH to LOW** learning rates to prioritize early-stopping cases:
- Actor LRs: `[0.1, 0.01, 0.001, 0.0001]`
- Critic LRs: `[0.1, 0.01, 0.001, 0.0001]`
- Total: **16 experiments**

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted!")

## Cell 2: Setup - Choose Start Mode

The experiment script **automatically restores from Google Drive** when you run it. Choose how to proceed:

- **Option 1: Continue** - Run experiments (auto-restores completed ones from Drive)
- **Option 2: Start Fresh** - Clear ALL results (local + Drive) and start from scratch  
- **Option 3: Import from Zip** - Import results from a downloaded zip file

In [ ]:
import os
import zipfile
import shutil
import json

EXPERIMENT_NAME = 'adaptive_scaling_experiment'
EXPERIMENT_ZIP = f'{EXPERIMENT_NAME}.zip'
RESULTS_DIR = f'/content/results/{EXPERIMENT_NAME}'
EXPERIMENT_DIR = f'/content/{EXPERIMENT_NAME}'
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/adaptive_scaling_results'

# ============================================================
# STEP 1: Extract experiment files
# ============================================================
if not os.path.exists(f'{EXPERIMENT_DIR}/mec_env.py'):
    drive_zip = f'/content/drive/MyDrive/{EXPERIMENT_ZIP}'
    
    if os.path.exists(drive_zip):
        print(f"Found experiment zip in Drive: {drive_zip}")
        zip_path = drive_zip
    else:
        print(f"Upload {EXPERIMENT_ZIP}:")
        from google.colab import files
        uploaded = files.upload()
        zip_path = list(uploaded.keys())[0]
    
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content')
    print("Experiment files extracted!")
else:
    print("Experiment files already present!")

# ============================================================
# STEP 2: Check existing progress
# ============================================================
drive_status_file = os.path.join(DRIVE_BACKUP_DIR, 'experiment_status.json')
if os.path.exists(drive_status_file):
    with open(drive_status_file) as f:
        drive_status = json.load(f)
    drive_completed = len(drive_status.get('completed', []))
    in_progress = drive_status.get('in_progress')
    print(f"\n*** Found Drive backup: {drive_completed}/16 completed ***")
    if in_progress:
        print(f"    Last in progress: {in_progress} (will restart this one)")
else:
    drive_completed = 0
    print("\n*** No existing Drive backup found ***")

# ============================================================
# CHOOSE START MODE
# ============================================================
print("\n" + "=" * 60)
print("SELECT START MODE")
print("=" * 60)
print("1. Continue - Auto-restore from Drive and run remaining")
print("2. Start Fresh - Clear ALL results and start from scratch")
print("3. Import from Zip - Upload results zip to restore")
print("=" * 60)

start_mode = input("Enter choice (1/2/3): ").strip()

if start_mode == '2':
    print("\n" + "=" * 60)
    print("STARTING FRESH - Clearing all results")
    print("=" * 60)
    if os.path.exists(RESULTS_DIR):
        shutil.rmtree(RESULTS_DIR)
        print(f"Cleared: {RESULTS_DIR}")
    if os.path.exists(DRIVE_BACKUP_DIR):
        shutil.rmtree(DRIVE_BACKUP_DIR)
        print(f"Cleared: {DRIVE_BACKUP_DIR}")
    print("Ready to run all 16 experiments from scratch!")

elif start_mode == '3':
    print("\n" + "=" * 60)
    print("IMPORT FROM ZIP")
    print("=" * 60)
    
    print("Upload your results zip file:")
    from google.colab import files
    uploaded = files.upload()
    results_zip_path = list(uploaded.keys())[0]
    
    temp_dir = '/content/temp_import'
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir)
    
    with zipfile.ZipFile(results_zip_path, 'r') as z:
        z.extractall(temp_dir)
    
    # Find experiment_status.json
    actual_results = temp_dir
    for root, dirs, files_list in os.walk(temp_dir):
        if 'experiment_status.json' in files_list:
            actual_results = root
            break
    
    # Copy to Drive backup directory (so auto-restore picks it up)
    if os.path.exists(DRIVE_BACKUP_DIR):
        shutil.rmtree(DRIVE_BACKUP_DIR)
    shutil.copytree(actual_results, DRIVE_BACKUP_DIR)
    
    with open(os.path.join(DRIVE_BACKUP_DIR, 'experiment_status.json')) as f:
        imported_status = json.load(f)
    print(f"Imported {len(imported_status.get('completed', []))} completed experiments to Drive")
    print("The experiment script will auto-restore these when you run it.")
    
    shutil.rmtree(temp_dir)

else:
    print("\n" + "=" * 60)
    print("CONTINUE MODE")
    print("=" * 60)
    if drive_completed > 0:
        print(f"Will auto-restore {drive_completed} completed experiments from Drive")
    print("Ready to run!")

print("\n" + "=" * 60)
print("SETUP COMPLETE - Run the next cell to start experiments")
print("=" * 60)

## Cell 3: Check GPU and Environment

In [ ]:
import torch
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("")
    print("WARNING: No GPU detected!")
    print("Go to: Runtime -> Change runtime type -> GPU")

# Set working directory
os.chdir('/content')
print(f"\nWorking directory: {os.getcwd()}")

## Cell 4: Verify Adaptive Scaling Architecture

In [ ]:
import sys
sys.path.insert(0, '/content/adaptive_scaling_experiment')

from Model import AdaptiveUniformTanhActor, CriticNetwork, ActorNetwork
import torch

# Compare architectures
original_actor = ActorNetwork(7, 3, torch.tanh)
adaptive_actor = AdaptiveUniformTanhActor(7, 3, target_range=4.0)
critic = CriticNetwork(350, 150, 7, 3)

print("=" * 60)
print("ARCHITECTURE COMPARISON")
print("=" * 60)
print(f"\nOriginal Actor: 7 -> 64 -> 32 -> tanh -> 3")
print(f"  Parameters: {sum(p.numel() for p in original_actor.parameters()):,}")
print(f"\nAdaptive Scaling Actor: 7 -> 64 -> 32 -> AdaptiveScale -> tanh -> 3")
print(f"  Parameters: {sum(p.numel() for p in adaptive_actor.parameters()):,}")
print(f"  Target Range: {adaptive_actor.target_range}")
print(f"\nCritic: 510 -> 512 -> 128 -> 1 (linear)")
print(f"  Parameters: {sum(p.numel() for p in critic.parameters()):,}")

print("\n" + "=" * 60)
print("Adaptive Scaling Mechanism:")
print("=" * 60)
print("  1. Track G_min, G_max of pre-activations (strict min/max)")
print("  2. Scale [G_min, G_max] -> [-4, +4]")
print("  3. Apply tanh: tanh(±4) ≈ ±0.999")
print("\nResult: Uniform tanh output by default, ±1 only via genuine learning")

## Cell 5: Run All 16 Experiments

This will run all 16 experiments with **HIGH learning rates FIRST**:
- Actor LRs: `[0.1, 0.01, 0.001, 0.0001]` (high to low)
- Critic LRs: `[0.1, 0.01, 0.001, 0.0001]` (high to low)

**Progress is auto-saved to Google Drive every 100 episodes.**

In [ ]:
import sys
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/adaptive_scaling_experiment')

from run_adaptive_scaling_experiment import run_all_experiments

# Run all experiments
run_all_experiments()

## Cell 6: Check Experiment Status

In [ ]:
import json
import os

results_dir = '/content/results/adaptive_scaling_experiment'
status_file = os.path.join(results_dir, 'experiment_status.json')

if os.path.exists(status_file):
    with open(status_file) as f:
        status = json.load(f)
    print("Experiment Status:")
    print(f"  Completed: {len(status['completed'])}/16")
    if status['in_progress']:
        print(f"  In progress: {status['in_progress']}")
    print("\nCompleted experiments:")
    for exp in sorted(status['completed']):
        print(f"  - {exp}")
else:
    print("No status file found yet.")

# Also check Drive backup
drive_dir = '/content/drive/MyDrive/adaptive_scaling_results'
if os.path.exists(drive_dir):
    print(f"\nDrive backup exists: {drive_dir}")
    contents = os.listdir(drive_dir)
    print(f"  Contents: {contents[:5]}..." if len(contents) > 5 else f"  Contents: {contents}")

## Cell 7: View Results Summary

In [ ]:
import json
import os

results_dir = '/content/results/adaptive_scaling_experiment'

if os.path.exists(results_dir):
    print("Adaptive Scaling Experiment Results Summary:")
    print("="*70)
    print(f"{'Actor LR':<12} {'Critic LR':<12} {'Stop Ep.':<12} {'Final Reward':<15}")
    print("-"*70)
    
    results = []
    for exp_dir in sorted(os.listdir(results_dir)):
        result_file = os.path.join(results_dir, exp_dir, 'results.json')
        if os.path.exists(result_file):
            with open(result_file) as f:
                data = json.load(f)
            results.append(data)
    
    for data in sorted(results, key=lambda x: (-x['actor_lr'], -x['critic_lr'])):
        print(f"{data['actor_lr']:<12} {data['critic_lr']:<12} {data['stopping_episode']:<12} {data['final_reward']:<15.4f}")
    print("="*70)
else:
    print("No results directory found yet.")

## Cell 8: Compare with Original (No Adaptive Scaling) Experiment

In [ ]:
import json
import os

def load_results(results_dir):
    results = []
    if os.path.exists(results_dir):
        for exp_dir in os.listdir(results_dir):
            result_file = os.path.join(results_dir, exp_dir, 'results.json')
            if os.path.exists(result_file):
                with open(result_file) as f:
                    results.append(json.load(f))
    return results

adaptive_results = load_results('/content/results/adaptive_scaling_experiment')
original_results = load_results('/content/results/stopping_experiment')

# Also try Drive backups
if not original_results:
    original_results = load_results('/content/drive/MyDrive/gradient_asymmetry_results')

if original_results and adaptive_results:
    print("COMPARISON: Original vs Adaptive Scaling Actor")
    print("="*85)
    print(f"{'Actor LR':<10} {'Critic LR':<10} {'Original':<15} {'Adaptive':<15} {'Difference':<15}")
    print("-"*85)
    
    for orig in sorted(original_results, key=lambda x: (-x['actor_lr'], -x['critic_lr'])):
        adapt = next((r for r in adaptive_results 
                   if r['actor_lr'] == orig['actor_lr'] and r['critic_lr'] == orig['critic_lr']), None)
        if adapt:
            diff = adapt['stopping_episode'] - orig['stopping_episode']
            diff_str = f"+{diff}" if diff > 0 else str(diff)
            improved = "IMPROVED" if diff > 100 else ""
            print(f"{orig['actor_lr']:<10} {orig['critic_lr']:<10} {orig['stopping_episode']:<15} {adapt['stopping_episode']:<15} {diff_str:<10} {improved}")
    print("="*85)
elif adaptive_results:
    print("Original experiment results not found for comparison.")
else:
    print("No Adaptive Scaling results found yet. Run the experiment first.")

## Cell 9: Analyze Adaptive Scaling Statistics (G_min, G_max)

In [ ]:
import json
import os
import numpy as np

results_dir = '/content/results/adaptive_scaling_experiment'

if os.path.exists(results_dir):
    print("Adaptive Scaling Statistics (G_min, G_max evolution):")
    print("="*90)
    print(f"{'Actor LR':<10} {'Critic LR':<10} {'Avg G_min':<15} {'Avg G_max':<15} {'Avg G_range':<15} {'Saturation':<15}")
    print("-"*90)
    
    for exp_dir in sorted(os.listdir(results_dir)):
        tracking_file = os.path.join(results_dir, exp_dir, 'tracking_data.json')
        result_file = os.path.join(results_dir, exp_dir, 'results.json')
        if os.path.exists(tracking_file) and os.path.exists(result_file):
            with open(tracking_file) as f:
                tracking = json.load(f)
            with open(result_file) as f:
                result = json.load(f)
            
            scaling_hist = tracking.get('scaling_history', [])
            act_hist = tracking.get('activation_history', [])
            
            if scaling_hist:
                last = scaling_hist[-1]
                sat = act_hist[-1]['avg_actor_output_saturation'] if act_hist else 0
                print(f"{result['actor_lr']:<10} {result['critic_lr']:<10} {last['avg_g_min']:<15.4f} {last['avg_g_max']:<15.4f} {last['avg_g_range']:<15.4f} {sat:<15.2%}")
    print("="*90)
    print("\nNote: With adaptive scaling, G_min/G_max track observed pre-activation extremes.")
    print("The scaling maps [G_min, G_max] -> [-4, +4], so tanh(±4) ≈ ±0.999 at extremes.")
    print("This prevents accidental saturation while allowing convergence to bounds when needed.")
else:
    print("No results found yet.")

## Cell 10: Download Results

In [ ]:
import shutil
import os

results_dir = '/content/results/adaptive_scaling_experiment'
output_zip = '/content/adaptive_scaling_results.zip'

if os.path.exists(results_dir):
    shutil.make_archive('/content/adaptive_scaling_results', 'zip', results_dir)
    print(f"Created: {output_zip}")
    print(f"Size: {os.path.getsize(output_zip) / 1024:.1f} KB")
    
    # Download
    from google.colab import files
    files.download(output_zip)
else:
    print("No results directory found.")